# Algoritmos de Machine Learning (ML): El Enfoque Clásico en Biomedicina

¡Hola! En esta etapa del curso vamos a dejar de lado las Redes Neuronales Profundas (Deep Learning). Nos vamos a enfocar **exclusivamente en Machine Learning Clásico** (Árboles de Decisión, Máquinas de Vectores de Soporte, Regresiones Matemáticas).

El Machine Learning (ML) es increíblemente poderoso, más fácil de interpretar por los médicos y requiere mucho menos poder de cómputo.

En este blog interactivo, vamos a desglosar cómo aplicar ML clásico a las **dos grandes familias de datos en biomedicina**:
1. **Datos Estructurados (Tablas clínicas)**.
2. **Datos No Estructurados (Imágenes biomédicas)**... cómo hacer que un árbol "vea" una foto.

---

## RAMA 1: Machine Learning con Datos Estructurados (Tabulares)

Para este ejemplo, usaremos el **Dataset de Diabetes**. Contiene 10 variables de pacientes (Edad, Sexo, Índice de Masa Corporal - BMI, Presión Sanguínea y 6 mediciones de suero sanguíneo).

El objetivo es un problema de **Regresión**: Predecir matemáticamente *cuánto avanzará la enfermedad* (una métrica continua) un año después de estas pruebas.

### 1.1 Extracción y Carga de Expedientes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes

# Cargamos los datos clínicos
datos_diabetes = load_diabetes()
df_diab = pd.DataFrame(datos_diabetes.data, columns=datos_diabetes.feature_names)

# La variable objetivo: Cuánto progresó la enfermedad en 1 año
df_diab['Progresion_Enfermedad'] = datos_diabetes.target

print("---- Expediente Endocrinológico (Primeros 5 de 442 pacientes) ----")
print("Nota: Los datos clínicos aquí ya vienen previamente escalados biólogicamente entre -0.2 y 0.2 por los autores.")
display(df_diab.head())

### 1.2 Análisis Exploratorio (EDA)

Antes de aplicar los algoritmos, debemos entender visualmente qué signos vitales están impulsando la progresión de la diabetes.

In [ ]:
plt.figure(figsize=(14, 5))

# Gráfico 1: Impacto de la Masa Corporal
plt.subplot(1, 2, 1)
sns.regplot(data=df_diab, x='bmi', y='Progresion_Enfermedad', 
            scatter_kws={'alpha':0.5, 'color':'#2ca02c'}, line_kws={'color':'red'})
plt.title('Relación: Masa Corporal (BMI) vs Daño a un Año', fontsize=12)
plt.xlabel('Índice de Masa Corporal (Estandarizado)')
plt.ylabel('Gravedad de la Progresión')

# Gráfico 2: Mapa de Calor de Correlaciones
plt.subplot(1, 2, 2)
# Seleccionamos el top de características que más importan
top_cols = ['bmi', 'bp', 's3', 's5', 'Progresion_Enfermedad']
sns.heatmap(df_diab[top_cols].corr(), annot=True, cmap='Blues', fmt=".2f")
plt.title('Top Biomarcadores Correlacionados', fontsize=12)

plt.tight_layout()
plt.show()

### 1.3 Preprocesamiento y División

Separamos a los pacientes en grupo de estudio (Train) y grupo de prueba ciega (Test). Aprovechamos de escalar rápidamente.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_tab = df_diab.drop(columns=['Progresion_Enfermedad'])
y_tab = df_diab['Progresion_Enfermedad']

# 80% entrenamiento, 20% prueba
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_tab, y_tab, test_size=0.2, random_state=42)

# Estandarización clínica
scaler_t = StandardScaler()
X_train_t = scaler_t.fit_transform(X_train_t)
X_test_t = scaler_t.transform(X_test_t)

print("Preprocesamiento finalizado. Listos para entrenar algoritmos de Regresión.")

### 1.4 Entrenamiento y Examen en UCI (Métricas)

Llamamos a 3 éxpertos del ML Clásico en Regresión:
- **Regresión de Cresta (Ridge)**: Ajuste de líneas estadísticas.
- **Soporte Vectorial (SVR)**: Analiza hiperplanos óptimos.
- **Bosque Clínico (RandomForestRegressor)**: Árboles de decisión médica votando en conjunto.

Al ser regresión predictiva, no medimos "Falsos Negativos", medimos cuánto nos equivocamos en números absolutos (Error Cuadrático Medio - MSE) y cuánta precisión estructural tenemos (R-Cuadrado -> 1.0 es perfección).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

modelos_tabulares = {
    'Regresión de Cresta (Ridge)': Ridge(alpha=1.0),
    'Soporte Vectorial (SVR)': SVR(kernel='rbf', C=10),
    'Bosque Clínico (Random Forest)': RandomForestRegressor(n_estimators=100, random_state=42)
}

print("===== REPORTE DE EXÁMENES ENDOCRINOS (Récord ML Clásico) =====")
for nombre_modelo, modelo in modelos_tabulares.items():
    # Entrenamiento
    modelo.fit(X_train_t, y_train_t)
    
    # Evaluación en Test
    prediccion = modelo.predict(X_test_t)
    
    # Calificaciones
    r2 = r2_score(y_test_t, prediccion)
    mse = mean_squared_error(y_test_t, prediccion)
    
    print(f"{nombre_modelo}:")
    print(f"  - Precisión de Ajuste (R2): {r2:.3f}")
    print(f"  - Error de Desviación (MSE): {mse:.1f}\n")

---

## RAMA 2: Machine Learning con Datos No Estructurados (Imágenes)

¿Se puede hacer visión artificial SIN Deep Learning?
¡Sí!

Para este caso usaremos una base clásica de microscopía óptica de bajísima resolución (`8x8` píxeles) que simulan la morfología de 10 familias distintas de células. El problema es: **El Machine Learning Clásico no acepta matrices bidimensionales (Fotos). Exige listas tabulares.**

### 2.1 Carga de Imágenes de Microscopio

In [ ]:
from sklearn.datasets import load_digits

# Cargamos nuestras "células" de 8x8 píxeles
scans = load_digits()
imagenes = scans.images    # Son 1797 registros biológicos visuales
etiquetas = scans.target   # 10 tipos celulares (etiquetas del 0 al 9)

print(f"¡Extraídas {len(imagenes)} radiografías microscópicas!")
print(f"Tamaño nativo de cada biología visual: {imagenes.shape[1]}x{imagenes.shape[2]} píxeles.")

### 2.2 Inspección Visual del Patólogo

Echemos un vistazo humano a 5 de estas muestras crudas antes de intentar predecirlas matemáticamente.

In [ ]:
plt.figure(figsize=(12, 4))
for index, (imagen, etiqueta) in enumerate(zip(imagenes[0:5], etiquetas[0:5])):
    plt.subplot(1, 5, index + 1)
    # Imprimimos la matriz numérica transformada en escala de grises (óptico)
    plt.imshow(imagen, cmap=plt.cm.binary, interpolation='nearest')
    plt.title(f'Familia Célula: {etiqueta}', fontsize=12)
    plt.axis('off')

plt.suptitle('Muestras extraídas del Microscopio Óptico', fontsize=16)
plt.show()

### 2.3 Preprocesamiento Mágico: El Flattening (Aplastamiento)

Como se mencionó, un Árbol de Decisión colapsará si le envías un cuadrado plano `(8, 8)`. 
Lo que haremos es "desenrollar" (Flatten) esa imagen. Transformaremos la foto cuadrada de 8x8 en una lista plana de **64 números (Columnas)**.

Así, el Árbol considerará que cada pixel individual es un "Síntoma Numérico Distinto" (Paciente 1: Pixel_0=0.0, Pixel_1=15.0... Pixel_63=12.0).

In [ ]:
numero_de_muestras = len(imagenes)

# --------- AQUÍ ESTÁ EL FLATTENING ---------
# Le decimos a python: Manten el numero de muestras, pero aplasta el resto de dimensiones e únelas (-1)
X_img = imagenes.reshape((numero_de_muestras, -1)) 
# ------------------------------------------

print(f"Antes del Flattening (Forma 3D): {imagenes.shape}")
print(f"Después del Flattening (Forma Tabular Plana 2D): {X_img.shape}")
print("\n¡La imagen se ha convertido en una tabla clínica de 64 columnas perceptuales!")

# Lo dividimos para evitar que el algoritmo memorice y se equivoque en nuevas placas médicas
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, etiquetas, test_size=0.3, random_state=42)

### 2.4 Entrenamiento de Algoritmos Oculares (ML)

Enviaremos estos 64 «síntomas lumínicos» a dos modelos clasicos altamente letales y verás cómo sin ser "Deep Learning con Convoluciones" adquieren visión rápidamente.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

modelos_vision = {
    'SVM Visual (Polinomial)': SVC(kernel='poly', C=10),
    'Bosque Óptico (Random Forest)': RandomForestClassifier(n_estimators=200, random_state=42)
}

predicciones_vision = {}

print("---- ENTRENANDO SOFTWARE OULAR ----")
for nombre, modelo in modelos_vision.items():
    modelo.fit(X_train_img, y_train_img)
    # Evaluamos diagnóstico visual
    predicciones_vision[nombre] = modelo.predict(X_test_img)
    print(f"[Ok] {nombre} aprendió a diferenciar morfologías celulares.")

### 2.5 Resultados Fotográficos: La Matriz de Confusión

¿Cómo saber a qué célula le erró la máquina visual? Con una Matriz de Múltiples Clases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for idx, (nombre, prediccion) in enumerate(predicciones_vision.items()):
    matriz = confusion_matrix(y_test_img, prediccion)
    sns.heatmap(matriz, annot=True, fmt="d", cmap="BuPu", cbar=False, ax=axes[idx])
    
    axes[idx].set_title(f'Auditoría Visual:\n{nombre}', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Célula Real (Laboratorio Humano)')
    axes[idx].set_xlabel('Célula Inferida por ML Clásico')

plt.tight_layout()
plt.show()

# Imprimiendo el reporte de puntería general de nuestro mejor motor visual clásico
print("\n=== REPORTE DE PUNTERÍA (SVM Visual Polinomial) ===")
print(classification_report(y_test_img, predicciones_vision['SVM Visual (Polinomial)']))

## Conclusión de Machine Learning Biomédico

- El **ML Clásico brilla masivamente en Datos Clínicos (Tablas)**, ayudando velozmente a pronosticar regresiones, como la gravedad futura de casos de Diabetes usando pocos recursos y siendo hiper-preciso en milisegundos.
- Aunque las redes convolucionales aplastan a todos en el procesamiento de **Imágenes**, es vital saber que el Machine Learning *(SVM, Bosques)* sí puede resolver tareas ópticas de precisión media si realizas **Flattening** (aplastando las matrices 2D a gigantescas filas de datos puros).

¡Te animo a jugar con las celdas, cambiar el área de las gráficas o modificar los algoritmos paramétricos para que notes las diferencias técnicas a tiempo real!